# Vector Operations: Buffer, Join, Dissolve, Overlay

This session covers four vector operations in GeoPandas. 
1. **Buffer** 
2. **Spatial join**
3. **Dissolve** 
4. **Overlay** 

By the end of this session students will be able to:

- Reproject and buffer point data, and state why buffering requires a projected CRS
- Perform a spatial join that attaches one layer's attributes to another based on location
- Dissolve polygons into a coarser geography and describe what happens to the attribute columns
- Run an overlay operation and explain how its output differs from a spatial join's


### Standard collab setup

In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily

## Buffer

***What's a buffer operation???***


A buffer is a polygon that extends a fixed distance outward from a geometry. 

The distance is interpreted in the units of the layer's CRS. A geographic CRS measures in degrees, so a buffer on unprojected data has no meaningful linear size and should not be used for measurement. 

GeoPandas warns about this but does not refuse, so the check is yours to make. 

**Recall before running.** What does `.buffer()` take as its argument, what are that argument's units, and what type of object does it return: a GeoDataFrame or a GeoSeries? 

In [ ]:
DATA_BASE_P2 = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data/part2"

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE_P2 = "../data/part2"

import geopandas as gpd

stations = gpd.read_file(f"{DATA_BASE_P2}/cbw_stations.geojson")
print(stations.crs)

naive_buffer = stations.buffer(0.01)
print(naive_buffer.area.head(3))

naive_buffer.plot()


**Expected result:** a `UserWarning` ("Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect.") followed by three area values near `0.00031`. GeoPandas performed the arithmetic as requested in the CRS's own unit. The `0.01` is 0.01 degrees, which is roughly 1.1 km north to south at this latitude and less east to west, so the buffers are not even a consistent size from one station to the next. The return type is a GeoSeries of polygons, one per input row.

The plot visualizes the buffers

**Recall before running.** What does `.to_crs()` take, and does it modify the GeoDataFrame in place or return a new one? 


The next cell reprojects to `EPSG:26918` (NAD83 / UTM zone 18N, in meters) and then buffers by 2000. 

Estimate the area in square kilometers that a 2 km buffer should report before you check the printed value.

In [ ]:
stations_p = stations.to_crs(26918)
buf_2km = stations_p.buffer(2000)

print(stations_p.crs)
print((buf_2km.area / 1e6).head(3))  # m^2 -> km^2

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
gpd.GeoSeries(buf_2km).plot(ax=ax, color="steelblue", alpha=0.4, edgecolor="steelblue")
stations_p.plot(ax=ax, color="darkred", markersize=4)
ax.set_title("2 km buffers around Chesapeake monitoring stations (EPSG:26918)")
plt.show()

**Expected result:** about `12.546` km² per station, slightly under pi times 2² (12.566) because `.buffer()` approximates a circle with a many-sided polygon. The gap is under 0.2%. 

The plot shows circular buffers around each station. `.to_crs()` returned a new GeoDataFrame; `stations` itself is unchanged.

**Your turn.** Using `stations_p`, buffer by 5000 m into a variable `buf_5km`. Print the first three buffer areas in km² and confirm they are close to pi times 5², about 78.5 km². Then filter `stations_p` to `station_type == "main"` and buffer that subset by 1000 m. How many buffer polygons result, and does that count match the number of `"main"` rows reported by `.value_counts()`?

In [ ]:
### Try it here:



## Spatial join

***First, what's a "join"?***

A **spatial join** treats spatial locations as the "keys" linking two datasets. For example, which county contains this point, even if there is no tabular county identifier on either layer. `gpd.sjoin()` performs the function. `predicate="within"` keeps a match when the left geometry lies entirely inside the right geometry.

**Recall before running.** `gpd.sjoin()` takes two GeoDataFrames and two keyword arguments that matter here. What does `how` control, and what does `predicate` control? Which layer's rows does the output have one row per, when `how="left"`?

**Before running.** `stations` is in `EPSG:4326`; `cbw_counties.geojson` loads as `EPSG:4269`. The two look alike on a map but are not numerically identical. 

Do we expect `gpd.sjoin()` refuse to run on mismatched CRSs, run silently, or something in between? The next cell does the reprojection first, so that question is answered in the expected-result note rather than by the output.

In [ ]:
counties = gpd.read_file(f"{DATA_BASE_P2}/cbw_counties.geojson")
print("stations crs:", stations.crs, " counties crs:", counties.crs)

stations_matched = stations.to_crs(counties.crs)
joined = gpd.sjoin(stations_matched, counties[["name", "state", "geometry"]], how="left", predicate="within")
print(joined.shape)
joined[["station_id", "name", "state_right"]].head()

**Expected result:** the join returns **167 rows**, one more than the 166 input stations. That is not an error. A left join keeps every row of `stations_matched`, and one station lies close enough to a shared county line that it falls within two county polygons, producing two output rows for that station.

On the CRS question: `sjoin` on mismatched inputs issues a `UserWarning` and proceeds. It is a notice, not a error, which is why the cell reprojects `stations` to `counties.crs` first rather than relying on the warning.

In [ ]:
# Look at the two findings the row-count mismatch above is hiding
dupes = joined[joined["station_id"].duplicated(keep=False)]
print(dupes[["station_id", "name", "state_right"]])

unmatched = joined[joined["name"].isna()]
print(unmatched[["station_id", "latitude", "longitude", "station_type", "waterbody"]])

**Expected result:** station `TF1.6` appears twice, once matched to Calvert County and once to Prince George's County, both in Maryland. It sits on the shared boundary; this is a real boundary case, not a data error. Station `TF3.1` matches no county: its point is in open water, outside every county polygon in the file. Its other attributes fit the pattern. `TF3.1` is one of the 23 stations that carry null `station_type`, `state`, and `waterbody` values in the source, so a station with incomplete metadata is also one a spatial join cannot place on land.

In [ ]:
# A second worked example: how many matched stations land in each state?
matched = joined[joined["name"].notna()]
matched["state_right"].value_counts()

**Expected result:** Maryland and Virginia dominate, with 83 and 66 matched stations respectively, and DC accounts for 17.

No rows appear for New York, Pennsylvania, West Virginia, or Delaware. `cbw_stations.geojson` covers tidal stations only, and those states' Chesapeake territory is non-tidal upstream watershed. 

`cbw_counties.geojson` covers all seven jurisdictions; the join reports only what the stations file contains.

**Exercise.** Re-run the `sjoin` above with `predicate="intersects"` instead of `predicate="within"`, keeping the same two GeoDataFrames and `how="left"`. Does the row count change from 167? Predict first: `intersects` is a looser test than `within` (a point exactly on a boundary intersects both sides), so would you expect more, fewer, or the same number of matched rows?

In [ ]:
## Try it here:



## Dissolve

`.dissolve(by=...)` merges every row that shares a value in the named column into one row, unioning their geometries into a single, possibly multi-part, shape. It is the GeoPandas counterpart of `arcpy.management.Dissolve`.



**Before running.** The county file's provenance notes list seven jurisdictions: VA (100), PA (43), MD (24), NY (20), WV (14), DE (3), DC (1). How many rows will `counties.dissolve(by="state")` produce, and how many columns will survive alongside `geometry`?

In [ ]:
states = counties.dissolve(by="state", as_index=False)
print(states.shape)

fig, ax = plt.subplots(figsize=(6, 6))
states.plot(ax=ax, edgecolor="white", linewidth=0.8, column="state", legend=False, cmap="tab10")
ax.set_title("205 counties dissolved to 7 states")
plt.show()

**Expected result:** `(7, 7)`. The 205 county rows collapse to 7 state rows, one per unique `state` value. The column count is not 3: the default `aggfunc="first"` keeps every attribute column (`state_fp`, `county_fp`, `geoid`, `name`, `name_long`) and fills each with the value from **whichever row came first** within the group.

**Implications?** Virginia's dissolved row reports `geoid` `51121` and `name` `Montgomery`, Montgomery County's own values, now present in a row meant to represent the entire state. `

"first"` is doing exactly what it says, but the result is easy to misread. 

The same mechanism is more dangerous for a summable quantity: a population column would be reported as the first county's value rather than a state total, with no warning.

When only the merged geometry is wanted, subset to the dissolve key and `geometry` before dissolving, so nothing remains to be misread.

In [ ]:
# A second worked example: subset to geometry-only before dissolving
states_geom_only = counties[["state", "geometry"]].dissolve(by="state", as_index=False)
print(states_geom_only.shape)
states_geom_only.head()

**Expected result:** `(7, 2)`, `state` and `geometry` only. Selecting `counties[["state", "geometry"]]` before `.dissolve()` leaves no other columns for `aggfunc="first"` to fill. When per-state attributes computed from the county rows are actually wanted, request them explicitly with an `aggfunc`, as in the next example.

In [ ]:
# A third worked example: keep a per-state count instead of just merging geometry
county_counts = counties.dissolve(by="state", aggfunc="count") ### This is where it's different
county_counts[["geoid"]].rename(columns={"geoid": "county_count"})

**Expected result:** the same 7 states, now with a `county_count` column alongside the dissolved geometry. `aggfunc="count"` aggregates every surviving column during the dissolve rather than taking the first value. Virginia's 100 and Delaware's 3 match `counties["state"].value_counts()`; a dissolve with an aggregation and a groupby answer the same question in two ways.

**Exercise.** Run `counties["state"].value_counts()` directly and confirm it matches the `county_count` column above, state for state. Then dissolve on a more granular key, `.dissolve(by="county_fp")`. How many rows result, and does that number make sense given that `county_fp` is a state-scoped code rather than a globally unique one? No solution is shown; run it.

In [ ]:
## Try it here:



## Overlay

Buffer and spatial join leave the original geometries intact: 
- buffer builds a new shape around a feature
- join attaches attributes without changing either layer's geometry

**Overlay** is different. `gpd.overlay()` computes new geometry from the intersection, union, or difference of two layers. 

This section uses Chesapeake Bay watershed restoration data: 34 documented dam removals (2012 to 2019) and the stream segments identified as reopened to fish passage as a result.



**A note on the data before the code runs.** The source shapefile for the reopened streams carries `DAM_NAME`, `STATE_ID`, and `NIDID` fields intended to link each stream segment to the specific dam whose removal reopened it. Every one of those fields is empty in the curated file (the `streams_opened` layer of `cbw_dam_removals.gpkg`), a consequence of how the original shapefile's join was built. Only `removal_year` survived on both layers. The overlay below therefore asks a different, defensible question: not which streams a particular dam reopened, but how much of the reopened network lies within a given distance of any documented removal site.

### Let's try it:

**Two datasets, one file.** The dam-removal points and the reopened-stream lines ship together in a single GeoPackage, `cbw_dam_removals.gpkg`, as two layers. A GeoPackage can hold several datasets of different geometry types in one file, which GeoJSON cannot.

 It is the same container built in Lab 3 with `CreateSQLiteDatabase` and `ExportFeatures`; here GeoPandas reads it directly.

**Recall before running.** Two methods are involved. What does `gpd.list_layers()` take and return? What does the `layer=` argument of `gpd.read_file()` do, and what happens if it is omitted on a file with more than one layer?

In [ ]:
## read file, then list layers and their geometry types

dams_gpkg = f"{DATA_BASE_P2}/cbw_dam_removals.gpkg"
print(gpd.list_layers(dams_gpkg))        # which layers are in there, and their geometry types

In [ ]:
## read the two layers, reproject to a projected CRS, and create a 5 km buffer around each dam removal

dams = gpd.read_file(dams_gpkg, layer="dam_removals")
streams_opened = gpd.read_file(dams_gpkg, layer="streams_opened")

dams_p = dams.to_crs(26918)
streams_p = streams_opened.to_crs(26918)

recovery_zone = dams_p.copy()
recovery_zone["geometry"] = recovery_zone.buffer(5000)  # 5 km, in meters
recovery_zone = recovery_zone[["dam_name", "state", "removal_year", "geometry"]].rename(
    columns={"removal_year": "dam_removal_year"}
)

near_a_removal = gpd.overlay(streams_p, recovery_zone, how="intersection")
print(f"{len(near_a_removal)} stream segments intersect a 5 km recovery zone")

total_km = streams_p["length_km"].sum()
near_km = (near_a_removal.geometry.length / 1000).sum()
print(f"{near_km:.1f} of {total_km:.1f} km ({near_km / total_km:.1%}) of the reopened network")

**Expected result:** 870 stream segments survive the intersection, totaling roughly 503 of the network's 1,992 km, about **25%**. A quarter of the reopened network lies within 5 km of a mapped removal site; the other three quarters does not. 

***Note*** the form of `gpd.overlay()`'s output. The geometries in `near_a_removal` are clipped pieces of the original stream lines, cut at each buffer's edge. They are not the original segments with a flag attached (a spatial join would produce that), nor whole segments included because part of them touched a buffer. The clipping is done by the overlay.

**Exercise.** 

Change the buffer distance from 5000 to 10000 and re-run the overlay and the percentage calculation. Does the coverage roughly double, more than double, or less than double? Buffers around 34 separate points can overlap one another once they are large enough; account for that before you predict.

In [ ]:
### Try it here:



### A second worked example: overlay from the other direction

The overlay above asked a stream-centric question: how much of the reopened network lies near a removal site. 

`gpd.overlay()` does not privilege either input, so the same two ingredients (5 km buffers around the dam removals and a polygon layer) can answer a county-centric question instead: overlay the recovery zones against `cbw_counties.geojson` to find which counties carry the most restoration footprint.

In [ ]:
# Same two ingredients, overlay in the other direction: recovery zones onto counties
counties_p = counties.to_crs(26918)  # counties is still in EPSG:4269 from the join section
recovery_zone_only = recovery_zone[["dam_name", "dam_removal_year", "geometry"]]
counties_subset = counties_p[["name", "state", "geometry"]].rename(columns={"name": "county_name"})

zone_by_county = gpd.overlay(recovery_zone_only, counties_subset, how="intersection")
zone_by_county["area_km2"] = zone_by_county.geometry.area / 1e6
print(f"{len(zone_by_county)} overlay pieces across {zone_by_county[['county_name', 'state']].drop_duplicates().shape[0]} distinct counties")

county_totals = zone_by_county.groupby(["county_name", "state"])["area_km2"].sum().sort_values(ascending=False)
county_totals.head(5)

**Expected result:** 56 overlay pieces across 38 distinct counties out of 205, roughly a fifth of the file, and 34 buffers producing 56 pieces across 38 counties confirms that several counties are touched more than once. York County, PA leads with about **264.6 km²** of recovery-zone footprint, followed by Lancaster (**217.5 km²**), Potter and Blair (**156.8 km²** each), and Luzerne (**154.5 km²**), all in Pennsylvania. Five separate dam-removal buffers overlap York alone.

One caveat on the totals: `county_totals` sums each dam's piece separately, so where two buffers from different dams overlap within the same county, that shared area is counted twice. It occurs here, in 7 of the 34 buffer pairs. It is a small, known overcount, though worth noting.

## Summary

Four operations:

- `.buffer(distance)` returns a GeoSeries of polygons, one per input geometry, with `distance` in the CRS's units. Reproject to a projected CRS first.
- `gpd.sjoin(left, right, how=..., predicate=...)` attaches `right`'s attributes to `left`'s rows by a spatial test and leaves both geometries unchanged. Both inputs must share a CRS.
- `.dissolve(by=column, aggfunc=...)` merges rows that share a value, unions their geometries, and aggregates the remaining columns according to `aggfunc`, which defaults to `"first"`.
- `gpd.overlay(a, b, how=...)` computes new geometry from the two layers. For `how="intersection"`, the output pieces are clipped at the boundaries of both inputs.

Two methods for multi-layer files: `gpd.list_layers(path)` reports what a GeoPackage contains, and `gpd.read_file(path, layer=name)` selects one layer from it.

## Wrap-up and next steps

**Practice** :

1. Buffer `dams_p` by 1000 m instead of 5000 and re-run the overlay. How small must the buffer become before the reopened-network coverage drops below 10%? One or two additional distances are enough; the exact threshold is not required.
2. Dissolve `counties` by `state`, buffer the dissolved Virginia polygon by 10 km in a projected CRS, and plot it against the 100 undissolved Virginia counties.
3. Using `joined` from the spatial-join section, count the matched stations per `waterbody` value with `joined["waterbody"].value_counts()`.

**This week's reading:** On Canvas

**Lab 4** ("Vectors, Open Source: Chesapeake Stations") opens today

**Next week:** no class meetings. Use the week to make progress on Lab 4. An optional extra-credit assignment, reproducing Practicum 1 entirely with open-source tools, will be posted on Canvas for the same week. 

**Week 11:** raster analysis with rasterio